In [ ]:
# 获取fusion candles

from research.utils import FusionCandles

# 设定训练集范围
START = "2024-06-01"
END = "2025-06-01"

candle_fetcher = FusionCandles(
    exchange="Binance Perpetual Futures", symbol="BTC-USDT", timeframe="1m"
)
candles = candle_fetcher.get_candles(START, END)
candles.shape

In [ ]:
# 制作标签
from research.labeler.gmm_labeler import GMMLabeler

LOG_RETURN_LAG = 4
LABEL_TYPE = "hard"

labeler = GMMLabeler(candles, lag_n=LOG_RETURN_LAG, verbose=False)

if LABEL_TYPE == "hard":
    # 分类模型标签
    raw_labels = labeler.label_hard_state
else:
    # 回归模型标签
    raw_labels = labeler.label_direction_force

raw_labels.shape

In [ ]:
# 制作特征
import pandas as pd
from src.features.simple_feature_calculator import SimpleFeatureCalculator
from src.features.simple_feature_calculator.buildin.feature_names import BUILDIN_FEATURES

FEATURE_NAMES = list(BUILDIN_FEATURES)[:100]  # 简化以节约演示时间

calc = SimpleFeatureCalculator(verbose=True)
calc.load(candles, sequential=True)

features_dict = calc.get(FEATURE_NAMES)
global_features = pd.DataFrame(features_dict)
print(global_features.shape)
global_features.head(1)

In [ ]:
# 对齐label与feature
from research.utils import align_features_labels

PRED_NEXT = 3

features, labels = align_features_labels(
    global_features,
    raw_labels,
    log_return_lag=LOG_RETURN_LAG,
    pred_next=PRED_NEXT,
)

assert len(features) == len(labels)
print(f"对齐后样本数: {len(labels)}")

In [ ]:
# 特征筛选
from src.features.feature_selection import GrootCVConfig, GrootCVSelector

GROOTCV_CUTOFF = 3

groot_cv_config = GrootCVConfig(cutoff=GROOTCV_CUTOFF)
selector = GrootCVSelector(config=groot_cv_config, verbose=True)
selector.fit(features, labels)

selected_features = selector.selected_features_
print(selected_features)